# STAT 764 · Lab 3 — The number you would actually report

**Assigned Tue Sep 22 · Due Tue Sep 29, 11:59 pm**

Individual work. Parts 1 and 2 need Meeting 5 (Tuesday Sep 22); Part 3 leans on
Meeting 4, and question **(d)** needs the reading.

Same rules as before: `git pull`, copy into `work/` before editing, Restart &
Run All before submitting, upload the `.ipynb` to Canvas.

A theme you should expect by now: nothing here is about making a model better.
Every part is about what you would put in front of someone who has to act on it.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ames = load("ames.csv")
y = ames["SalePrice"]
print(f"{len(ames)} sales, {ames.shape[1]} columns")

---

## Part 1 · Three numbers, and the one you would report (15 points)

Pick your own predictors — at least four numeric and at least one categorical,
every preprocessing step inside the pipeline, as in Lab 1. Split with
`test_size=0.25, random_state=764`.

Report, on held-out data, for **both** the mean baseline and your pipeline:

- R-squared
- RMSE, in dollars
- MAE, in dollars

**(a)** A homeowner asks how far off your estimate of *their* house will be.
Which of your three numbers answers that question, and what is it? One sentence
in the written section.

⚠ The baseline is not a formality. If your pipeline does not beat it on the
metric you chose, say so rather than quietly switching metrics.

In [ ]:
# YOUR CODE HERE

## Part 2 · The scale is a decision (20 points)

Fit **the same pipeline** twice: once on `SalePrice`, once on
`np.log(SalePrice)` with the predictions transformed back to dollars with
`np.exp`. Report all three metrics for both.

In Meeting 5 this moved R-squared and RMSE one way and MAE the other. You are
checking whether that holds for *your* column choice.

**Then do it again on at least ten more splits.** On Meeting 6 we measured this
over thirty: log beat plain OLS on **MAE 29 times out of 30**, and on **RMSE 15
times out of 30.** One of those is a property of the transformation. The other is
a coin flip, and reporting it from a single split is reporting the draw.

So for each of your three metrics, report **how often** the log model won, not
just whether it won on `random_state=764`:

```python
for seed in range(10):                  # or more
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    ...                                 # fit both, score both, record the winner
```

**(b)** Two mechanisms were named in class for why logging changes the answer.
Name whichever one you think dominates **for your model**, point at a number in
your own output that supports it, and say **which of your three metrics you
actually trust to rank the two models** — given what your seed loop showed. If
your result disagrees with the meeting's, say so; that is a finding, not a
mistake.

⚠ Do not report `r2_score` on the log scale and compare it with R-squared in
dollars. They are not the same quantity. Everything gets compared in dollars.

In [ ]:
# YOUR CODE HERE

## Part 3 · What actually makes a leak big? (15 points)

Meeting 4 measured a structural leak worth exactly `0.000000`: standardising on
the full dataset cannot change an OLS fit. Meeting 5 raised a second one —
imputing from the full dataset — and this part measures it properly.

Add `Lot_Frontage` to your predictors. It is missing in **16.7%** of rows. You
will build the same comparison four times.

**(i) At the missing rate it ships with.** Held-out MAE for:

| | |
|---|---|
| **honest** | `SimpleImputer` inside the pipeline — the median comes from training rows only |
| **leaky** | fill with the median of the **full dataset**, then split |

**(ii) Now blank most of it.** Set `Lot_Frontage` to `NaN` for a random ~95% of
rows, and repeat (i). Use a seeded generator so your result is reproducible:

```python
rng = np.random.default_rng(764)
blanked = ames.copy()
blanked.loc[rng.random(len(blanked)) < 0.95, "Lot_Frontage"] = np.nan
```

**(iii) Change what the filled-in value is computed from.** Instead of the
column's own median, fill each hole with the median `Lot_Frontage` of houses in
the **same price decile** — which uses `SalePrice`, the thing you are predicting:

```python
band = pd.qcut(ames["SalePrice"], 10, labels=False)
leaky = ames.copy()
leaky["Lot_Frontage"] = leaky["Lot_Frontage"].fillna(
    leaky.groupby(band)["Lot_Frontage"].transform("median"))
```

Do this at **both** missing rates — the original and the blanked one.

Report all of it as one small table: missing rate, what the imputation was
computed from, MAE, and the gap against the honest version.

⚠ Two things changed across those four fits: **how much was missing**, and
**what the filled-in value was computed from.** Only one of them moves the gap.
Part 4(c) asks you which — do not guess, read it off your own table.

In [ ]:
# YOUR CODE HERE


# ## Part 4 · Written (25 points)
#
# Three or four sentences each. Padding will not help you.
#
# **(a)** From Part 1: which number answers the homeowner's question, and what is it?
#
# **(b)** From Part 2: which mechanism dominates for your model, and which number
# in your output says so?
#
# **(c)** From Part 3: two things changed across your four fits — how much was
# missing, and what the filled-in value was computed from. **Which one moved the
# gap, and why?** Then connect it to the leak you measured in Lab 2, which was
# worth 0.11 of R-squared: what do that leak and your largest gap here have in
# common that the others do not?
#
# **(d)** 📗 **From the reading.** ESL §7.4 gives the optimism of the training
# error rate. It says the expected optimism depends on a specific quantity
# computed from your fitted values and your outcomes. **Name that quantity**, and
# say what it implies about adding a predictor that is pure noise — does the
# training error go up, down, or stay the same, and what happens to the optimism?
#
# ⚠ The answer to (d) is in the reading and *not* in any notebook.
#
# **(e)** You now have several numbers for one model and no rule for choosing
# between them. Write the two sentences you would put at the top of a report to a
# client, naming your number and what choosing it hides.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

**(e)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 reports all three metrics for baseline *and* pipeline
- [ ] Part 2 compares everything **in dollars**
- [ ] Part 3 reports all four fits in one table
- [ ] All five written answers are filled in
- [ ] Uploaded the `.ipynb` to Canvas

**75 points.**